# Real-Time Voice Transformer (PyAudio Version)

It looks like the `sounddevice` module's underlying audio engine (PortAudio) is struggling to find your Windows devices. We are switching to `PyAudio`, which is the industry standard for real-time audio and often more reliable on Windows.

Ensure you have `pyaudio` and `numpy` installed:
```bash
pip install pyaudio numpy scipy
```

In [ ]:
import pyaudio
import numpy as np
import traceback

# Settings
SAMPLE_RATE = 44100
CHUNK_SIZE = 2048
CHANNELS = 1
TRANSFORM_TYPE = 'robot' # Options: 'robot', 'pitch_shift_down', 'pitch_shift_up', 'none'

def process_audio(audio_chunk, frames):
    if TRANSFORM_TYPE == 'robot':
        # Simple robot effect via amplitude modulation
        t = np.arange(frames) / SAMPLE_RATE
        modulator = 0.5 * (1 + np.sin(2 * np.pi * 50 * t))
        processed = audio_chunk * modulator
        
    elif TRANSFORM_TYPE == 'pitch_shift_down':
        indices = np.arange(0, frames, 0.75)
        indices = indices[indices < frames].astype(int)
        processed = np.zeros(frames)
        processed[:len(indices)] = audio_chunk[indices]
        
    elif TRANSFORM_TYPE == 'pitch_shift_up':
        indices = np.arange(0, frames, 1.5)
        indices = indices[indices < frames].astype(int)
        processed = np.zeros(frames)
        processed[:len(indices)] = audio_chunk[indices]
        
    else:
        processed = audio_chunk
        
    return processed

p = pyaudio.PyAudio()

# Check available devices just to be safe
print("--- Audio System Diagnostics ---")
device_count = p.get_device_count()
print(f"PyAudio found {device_count} audio devices.")
if device_count == 0:
    print("\nWAIT: PyAudio also sees ZERO audio devices. This almost always means:")
    print("1. Windows 'Microphone Privacy Settings' is blocking python/VS Code from accessing the mic.")
    print("2. Or, you are running this in an environment without audio (like WSL or Docker).")

print(f"\nAttempting to start stream with {TRANSFORM_TYPE}...")

stream = None
try:
    stream = p.open(format=pyaudio.paFloat32,
                    channels=CHANNELS,
                    rate=SAMPLE_RATE,
                    input=True,
                    output=True,
                    frames_per_buffer=CHUNK_SIZE)
    
    print("\n--- STREAM STARTED SUCCESSFULLY ---")
    print("Speak into your microphone. Stop the Jupyter kernel to end.")
    
    while True:
        data = stream.read(CHUNK_SIZE, exception_on_overflow=False)
        audio_chunk = np.frombuffer(data, dtype=np.float32)
        
        processed = process_audio(audio_chunk, CHUNK_SIZE)
        
        # Write processed audio back out
        stream.write(processed.astype(np.float32).tobytes())
        
except KeyboardInterrupt:
    print("\nStream stopped by user.")
except Exception as e:
    print("\nError starting/running stream:", e)
    traceback.print_exc()
    print("\n--- PyAudio device list ---")
    for i in range(p.get_device_count()):
        print(i, p.get_device_info_by_index(i).get('name'))
    print("\nIf your default device is failing, look at the list above and set input_device_index=ID and output_device_index=ID inside p.open(...)")
finally:
    if stream is not None:
        stream.stop_stream()
        stream.close()
    p.terminate()